## Surrogate Model Extension: Neutron Transmission

This extension follows the latest main neutron-transport notebook. The Monte Carlo slab simulation is accurate but computationally expensive: each call to the transport simulator runs thousands of neutron histories. A **surrogate model** (also called an emulator) is a fast approximating function trained on simulation data. Once trained, it can predict transmission rates for new inputs in microseconds rather than seconds.

The surrogate takes three physically meaningful inputs:

$$\mathbf{x} = (\Sigma_a,\ \Sigma_s,\ L)$$

and predicts the transmitted fraction $T$. The macroscopic cross-sections $\Sigma_a$ and $\Sigma_s$ fully determine the material's interaction physics, while $L$ sets the geometry. This is a more general parameterisation than using named materials, because it allows interpolation and extrapolation to hypothetical materials not in the original dataset.

Two implementations are compared:
1. **sklearn** `MLPRegressor` - rapid prototyping with automatic training loop.
2. **PyTorch** feedforward network - explicit control over architecture and training, closer to how SciML models are built in practice.

Both are trained on the same dataset and evaluated against held-out Monte Carlo results.

### Dependencies

This notebook uses `numpy`, `matplotlib`, `scikit-learn`, and `torch`. These packages are listed in the repository `requirements.txt`, so the notebook avoids installing packages inside the analysis itself.

In [ ]:
# ---------------------------------------------------------------------------
# Imports for the surrogate model section
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch version: {torch.__version__}")
SURROGATE_RNG = np.random.default_rng(42)

# ---------------------------------------------------------------------------
# Standalone fallback helpers
# ---------------------------------------------------------------------------
# The helpers below mirror the latest main notebook closely enough for this
# extension to run either standalone or after the main analysis cells.
from dataclasses import dataclass


def isotropic_unit_vectors(rng, n):
    """Sample n isotropic 3D unit vectors."""
    if n == 0:
        return np.zeros((0, 3))
    mu = rng.uniform(-1.0, 1.0, n)
    phi = rng.uniform(0.0, 2.0 * np.pi, n)
    sin_theta = np.sqrt(1.0 - mu**2)
    return np.column_stack([
        sin_theta * np.cos(phi),
        sin_theta * np.sin(phi),
        mu,
    ])


def show_fig():
    """Display the current Matplotlib figure in notebooks and scripts."""
    plt.show()


@dataclass(frozen=True)
class Material:
    name: str
    macro_absorption: float
    macro_scattering: float


if "materials" not in globals():
    materials = {
        "Water": Material("Water", 0.022236, 3.442996),
        "Lead": Material("Lead", 0.005212, 0.370151),
        "Graphite": Material("Graphite", 0.000377, 0.396877),
    }


### Extension 1 - Dataset Generation

Training data are generated by random Latin-hypercube-style sampling over a physically motivated parameter space. The ranges for $\Sigma_a$ and $\Sigma_s$ bracket the three materials already studied (Water, Lead, Graphite) and extend beyond them to give the surrogate some extrapolation coverage.

Each sample is an independent Monte Carlo estimate of $T$, so it carries statistical noise proportional to $1/\sqrt{N}$. This is realistic: a surrogate trained on noisy simulation data must learn the underlying trend, not the noise.

In [ ]:
# ---------------------------------------------------------------------------
# Parameter space (physically motivated ranges)
# Water:    Sigma_a ~ 0.022236, Sigma_s ~ 3.442996
# Lead:     Sigma_a ~ 0.005212, Sigma_s ~ 0.370151
# Graphite: Sigma_a ~ 0.000377, Sigma_s ~ 0.396877
# We sample broadly so the surrogate is not just interpolating three points.
# ---------------------------------------------------------------------------
SIGMA_A_RANGE = (1e-4, 0.05)   # cm^-1
SIGMA_S_RANGE = (0.01, 5.0)    # cm^-1
L_RANGE       = (1.0,  30.0)   # cm
N_TRAIN_SIMS  = 800             # number of MC runs for training
N_PER_SIM     = 1500            # neutrons per MC run for surrogate data


def sample_material_and_thickness(rng, n):
    """Latin-hypercube-style uniform sampling over (Sigma_a, Sigma_s, L)."""
    sigma_a = rng.uniform(*SIGMA_A_RANGE, n)
    sigma_s = rng.uniform(*SIGMA_S_RANGE, n)
    L       = rng.uniform(*L_RANGE, n)
    return sigma_a, sigma_s, L


def simulate_from_cross_sections(rng, sigma_a, sigma_s, L, n_neutrons):
    """
    Run simulate_slab for a material defined directly by its macroscopic
    cross-sections rather than by a named Material dataclass.
    Returns (transmission_fraction, uncertainty).
    """
    import math
    sigma_t = sigma_a + sigma_s
    lambda_t = 1.0 / sigma_t
    p_absorb = sigma_a / sigma_t

    positions  = np.zeros((n_neutrons, 3))
    directions = np.zeros((n_neutrons, 3))
    directions[:, 0] = 1.0
    n_transmitted = 0

    for _ in range(50_000):
        active = len(positions)
        if active == 0:
            break
        steps     = rng.exponential(lambda_t, active)
        positions = positions + steps[:, None] * directions

        reflected    = positions[:, 0] < 0.0
        transmitted  = positions[:, 0] > L
        n_transmitted += int(transmitted.sum())

        inside    = ~(reflected | transmitted)
        positions = positions[inside]
        directions = directions[inside]
        if len(positions) == 0:
            break

        absorbed   = rng.uniform(size=len(positions)) < p_absorb
        surviving  = ~absorbed
        positions  = positions[surviving]
        directions = isotropic_unit_vectors(rng, len(positions))

    T = n_transmitted / n_neutrons
    sigma_T = math.sqrt(T * (1.0 - T) / n_neutrons)
    return T, sigma_T


print("Generating training dataset - this takes ~30-60 s ...")
sigma_a_samples, sigma_s_samples, L_samples = sample_material_and_thickness(
    SURROGATE_RNG, N_TRAIN_SIMS
)

T_values  = np.zeros(N_TRAIN_SIMS)
T_sigmas  = np.zeros(N_TRAIN_SIMS)

for i in range(N_TRAIN_SIMS):
    T_values[i], T_sigmas[i] = simulate_from_cross_sections(
        SURROGATE_RNG,
        sigma_a_samples[i],
        sigma_s_samples[i],
        L_samples[i],
        N_PER_SIM,
    )
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{N_TRAIN_SIMS} simulations done")

X = np.column_stack([sigma_a_samples, sigma_s_samples, L_samples])
y = T_values

print(f"\nDataset shape: {X.shape}")
print(f"Transmission range: [{y.min():.3f}, {y.max():.3f}]")

### Extension 2 - Exploratory Data Analysis

Before fitting a model it is useful to verify that the generated data are physically sensible. Transmission should decrease with thickness and increase with lower total cross-section.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
labels = [r"$\Sigma_a$ / cm$^{-1}$", r"$\Sigma_s$ / cm$^{-1}$", "Thickness $L$ / cm"]

for ax, xi, label in zip(axes, X.T, labels):
    sc = ax.scatter(xi, y, c=T_sigmas, cmap="viridis", s=10, alpha=0.6)
    ax.set_xlabel(label)
    ax.set_ylabel("Transmission $T$")
    ax.set_title(f"$T$ vs {label.split('/')[0].strip()}")
    plt.colorbar(sc, ax=ax, label=r"$\sigma_T$")

plt.suptitle("Training data: transmission vs input features (colour = MC uncertainty)")
plt.tight_layout()
show_fig()

The scatter plots confirm the expected trends. Transmission decreases monotonically with thickness. The dependence on $\Sigma_s$ is strong: materials with large scattering cross-sections produce many reflected neutrons, reducing transmission sharply. The MC uncertainty (colour scale) is larger when $T$ is near $0.5$ and smaller at the extremes, consistent with the binomial formula $\sigma_T = \sqrt{T(1-T)/N}$.

### Extension 3 - Surrogate Model I: sklearn MLPRegressor

The inputs are first standardised to zero mean and unit variance, which is essential for gradient-based training of neural networks. The sklearn `MLPRegressor` uses `adam` by default and handles the training loop automatically.

In [ ]:
# ---------------------------------------------------------------------------
# Train/test split and input standardisation
# ---------------------------------------------------------------------------
X_train, X_test, y_train, y_test, sig_train, sig_test = train_test_split(
    X, y, T_sigmas, test_size=0.2, random_state=0
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ---------------------------------------------------------------------------
# Train sklearn MLP
# ---------------------------------------------------------------------------
sklearn_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=2000,
    random_state=0,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    verbose=False,
)
sklearn_mlp.fit(X_train_sc, y_train)

y_pred_sk = sklearn_mlp.predict(X_test_sc)

# Clip to physical range [0, 1]
y_pred_sk = np.clip(y_pred_sk, 0.0, 1.0)

rmse_sk = np.sqrt(mean_squared_error(y_test, y_pred_sk))
r2_sk   = r2_score(y_test, y_pred_sk)

print("sklearn MLPRegressor results")
print(f"  Training iterations : {sklearn_mlp.n_iter_}")
print(f"  Test RMSE           : {rmse_sk:.4f}")
print(f"  Test R²             : {r2_sk:.4f}")

# Compare RMSE to average MC uncertainty in test set
mean_mc_uncertainty = sig_test.mean()
print(f"  Mean MC uncertainty : {mean_mc_uncertainty:.4f}")
print(f"  RMSE / MC uncertainty: {rmse_sk / mean_mc_uncertainty:.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Parity plot
ax = axes[0]
ax.errorbar(y_test, y_pred_sk, xerr=sig_test, fmt="o", markersize=4,
            alpha=0.5, elinewidth=0.7, capsize=2, label="test samples")
lims = [0, 1]
ax.plot(lims, lims, "k--", linewidth=1, label="perfect prediction")
ax.set_xlabel("Monte Carlo $T$ (true)")
ax.set_ylabel("Surrogate $\\hat{T}$ (sklearn)")
ax.set_title(f"sklearn MLP parity plot  ($R^2={r2_sk:.3f}$)")
ax.legend()

# Residual plot
ax = axes[1]
residuals = y_pred_sk - y_test
ax.scatter(y_test, residuals, s=12, alpha=0.55, color="#2a6fbb")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.fill_between([0, 1], -mean_mc_uncertainty, mean_mc_uncertainty,
                alpha=0.15, color="gray", label="mean MC uncertainty band")
ax.set_xlabel("Monte Carlo $T$ (true)")
ax.set_ylabel("Residual $\\hat{T} - T$")
ax.set_title("sklearn MLP residuals")
ax.legend()

plt.tight_layout()
show_fig()

### Extension 4 - Surrogate Model II: PyTorch Feedforward Network

The same architecture is reimplemented in PyTorch. This makes the training loop explicit and allows easy modification of the loss function, optimiser, and learning rate schedule - features that are important in physics-informed variants of neural networks.

In [ ]:
# ---------------------------------------------------------------------------
# Convert to PyTorch tensors
# ---------------------------------------------------------------------------
X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_t = torch.tensor(y_train,    dtype=torch.float32).unsqueeze(1)
X_test_t  = torch.tensor(X_test_sc,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,     dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)


# ---------------------------------------------------------------------------
# Network definition - identical structure to the sklearn model
# ---------------------------------------------------------------------------
class TransmissionNet(nn.Module):
    """Feedforward network for neutron transmission prediction.
    
    Input : (Sigma_a, Sigma_s, L) - standardised
    Output: transmission fraction T in [0, 1]
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),   # ensures output is in (0,1) - respects physical constraint
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)
model     = TransmissionNet()
optimiser = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=20, factor=0.5)
criterion = nn.MSELoss()

# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------
N_EPOCHS     = 300
train_losses = []
val_losses   = []

for epoch in range(N_EPOCHS):
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimiser.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimiser.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)
    train_losses.append(train_loss)

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_test_t), y_test_t).item()
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | train loss {train_loss:.5f} | val loss {val_loss:.5f}")

print("Training complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate PyTorch model
# ---------------------------------------------------------------------------
model.eval()
with torch.no_grad():
    y_pred_pt = model(X_test_t).squeeze().numpy()

rmse_pt = np.sqrt(mean_squared_error(y_test, y_pred_pt))
r2_pt   = r2_score(y_test, y_pred_pt)

print("PyTorch TransmissionNet results")
print(f"  Test RMSE            : {rmse_pt:.4f}")
print(f"  Test R²              : {r2_pt:.4f}")
print(f"  RMSE / MC uncertainty: {rmse_pt / mean_mc_uncertainty:.2f}x")

# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(train_losses, label="train loss", linewidth=1.2)
ax.plot(val_losses,   label="validation loss", linewidth=1.2)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("PyTorch training curves")
ax.set_yscale("log")
ax.legend()

ax = axes[1]
ax.errorbar(y_test, y_pred_pt, xerr=sig_test, fmt="o", markersize=4,
            alpha=0.5, elinewidth=0.7, capsize=2, label="test samples")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect prediction")
ax.set_xlabel("Monte Carlo $T$ (true)")
ax.set_ylabel("Surrogate $\\hat{T}$ (PyTorch)")
ax.set_title(f"PyTorch parity plot  ($R^2={r2_pt:.3f}$)")
ax.legend()

plt.tight_layout()
show_fig()

### Extension 5 - Model Comparison and Speed Benchmark

The key practical advantage of a surrogate model is speed. A single Monte Carlo run with $N=1500$ neutrons takes of order 1 s. The trained network evaluates in microseconds for a batch of the same size.

In [ ]:
import time

# ---------------------------------------------------------------------------
# Speed comparison on 100 new samples
# ---------------------------------------------------------------------------
N_BENCHMARK = 100
sa_b, ss_b, L_b = sample_material_and_thickness(SURROGATE_RNG, N_BENCHMARK)
X_bench    = np.column_stack([sa_b, ss_b, L_b])
X_bench_sc = scaler.transform(X_bench)

# MC timing
t0 = time.perf_counter()
T_mc_bench = np.array([
    simulate_from_cross_sections(SURROGATE_RNG, sa_b[i], ss_b[i], L_b[i], N_PER_SIM)[0]
    for i in range(N_BENCHMARK)
])
mc_time = time.perf_counter() - t0

# sklearn timing
t0 = time.perf_counter()
T_sk_bench = np.clip(sklearn_mlp.predict(X_bench_sc), 0.0, 1.0)
sk_time = time.perf_counter() - t0

# PyTorch timing
X_bench_t = torch.tensor(X_bench_sc, dtype=torch.float32)
model.eval()
t0 = time.perf_counter()
with torch.no_grad():
    T_pt_bench = model(X_bench_t).squeeze().numpy()
pt_time = time.perf_counter() - t0

print(f"Benchmark: {N_BENCHMARK} evaluations")
print(f"  Monte Carlo  : {mc_time:.2f} s  (reference)")
print(f"  sklearn MLP  : {sk_time*1000:.2f} ms  ({mc_time/sk_time:.0f}x speedup)")
print(f"  PyTorch MLP  : {pt_time*1000:.2f} ms  ({mc_time/pt_time:.0f}x speedup)")

rmse_sk_b = np.sqrt(mean_squared_error(T_mc_bench, T_sk_bench))
rmse_pt_b = np.sqrt(mean_squared_error(T_mc_bench, T_pt_bench))
print(f"\nBenchmark RMSE vs fresh MC")
print(f"  sklearn: {rmse_sk_b:.4f}")
print(f"  PyTorch: {rmse_pt_b:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Summary comparison figure
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Side-by-side parity: sklearn vs PyTorch on the benchmark set
for ax, y_pred, label, color in zip(
    axes,
    [T_sk_bench, T_pt_bench],
    [f"sklearn MLP  (RMSE={rmse_sk_b:.4f})", f"PyTorch MLP  (RMSE={rmse_pt_b:.4f})"],
    ["#2a6fbb", "#b23a48"],
):
    ax.scatter(T_mc_bench, y_pred, s=18, alpha=0.65, color=color, label=label)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_xlabel("Monte Carlo $T$ (benchmark)")
    ax.set_ylabel("Surrogate $\\hat{T}$")
    ax.set_title(label)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle("Surrogate model benchmark: both models vs fresh MC simulations")
plt.tight_layout()
show_fig()

# Speed bar chart
fig, ax = plt.subplots(figsize=(7, 4))
methods = ["Monte Carlo", "sklearn MLP", "PyTorch MLP"]
times_ms = [mc_time * 1000, sk_time * 1000, pt_time * 1000]
colors   = ["#888", "#2a6fbb", "#b23a48"]
bars = ax.bar(methods, times_ms, color=colors, alpha=0.82)
for bar, t in zip(bars, times_ms):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.03,
            f"{t:.1f} ms", ha="center", va="bottom", fontsize=11)
ax.set_yscale("log")
ax.set_ylabel(f"Wall time for {N_BENCHMARK} evaluations (ms, log scale)")
ax.set_title("Computational cost: Monte Carlo vs surrogate models")
show_fig()

### Extension 6 - Surrogate Predictions on a Thickness Scan

As a final validation, the surrogate is used to generate smooth transmission curves for the three named materials across the full thickness range. These curves can be compared directly to the Monte Carlo thickness scan in the main notebook.

In [ ]:
L_dense = np.linspace(1.0, 30.0, 200)

# Prefer the latest main notebook's thickness_results if this extension is run
# after those cells. Otherwise, generate a compact fallback MC reference scan.
if "scan_results" not in globals():
    scan_results = {}
    if "thickness_results" in globals():
        print("Using thickness_results from the main notebook for comparison ...")
        for material_name, results in thickness_results.items():
            uncertainties = results.get(
                "transmission_error",
                np.zeros_like(results["transmission"]),
            )
            scan_results[material_name] = [
                {
                    "L / cm": float(L_ref),
                    "transmitted fraction": float(T_ref),
                    "transmitted uncertainty": float(dT_ref),
                }
                for L_ref, T_ref, dT_ref in zip(
                    results["L"], results["transmission"], uncertainties
                )
            ]
    else:
        L_reference = np.linspace(1.0, 30.0, 8)
        print("Generating fallback MC scan_results for material comparison ...")
        for material in materials.values():
            rows = []
            for L_ref in L_reference:
                T_ref, dT_ref = simulate_from_cross_sections(
                    SURROGATE_RNG,
                    material.macro_absorption,
                    material.macro_scattering,
                    float(L_ref),
                    N_PER_SIM,
                )
                rows.append({
                    "L / cm": float(L_ref),
                    "transmitted fraction": T_ref,
                    "transmitted uncertainty": dT_ref,
                })
            scan_results[material.name] = rows

fig, ax = plt.subplots(figsize=(9, 5))

for material in materials.values():
    # --- Surrogate predictions (smooth curve) ---
    X_scan = np.column_stack([
        np.full(len(L_dense), material.macro_absorption),
        np.full(len(L_dense), material.macro_scattering),
        L_dense,
    ])
    X_scan_sc = scaler.transform(X_scan)
    X_scan_t  = torch.tensor(X_scan_sc, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        T_surrogate = model(X_scan_t).squeeze().numpy()

    line, = ax.plot(L_dense, T_surrogate, linewidth=2,
                    label=f"{material.name} (surrogate)")

    # --- MC scan points from the main notebook for comparison ---
    rows   = scan_results[material.name]
    L_mc   = np.array([row["L / cm"] for row in rows])
    T_mc   = np.array([row["transmitted fraction"] for row in rows])
    dT_mc  = np.array([row["transmitted uncertainty"] for row in rows])
    ax.errorbar(L_mc, T_mc, yerr=dT_mc, fmt="o", markersize=4,
                color=line.get_color(), alpha=0.7,
                label=f"{material.name} (MC scan)", capsize=2)

ax.set_xlabel("Slab thickness $L$ / cm")
ax.set_ylabel("Transmission fraction $T$")
ax.set_title("Surrogate model (lines) vs Monte Carlo scan (points) for all materials")
ax.legend(ncol=2, fontsize=9)
show_fig()

### Extension 7 - Discussion

Both surrogate models reproduce the Monte Carlo transmission data well across a broad parameter space, with RMSE comparable to the intrinsic Monte Carlo statistical uncertainty. The PyTorch implementation applies a sigmoid output activation which enforces the physical constraint $T \in (0, 1)$ by construction, rather than relying on post-hoc clipping. This is a simple example of **physics-constrained machine learning**: incorporating domain knowledge directly into the model architecture.

The key limitations of this surrogate are:
- **Training noise**: the labels $T$ are themselves stochastic, so the network learns a noisy version of the true response surface. Increasing $N$ per simulation would reduce label noise at the cost of longer data generation.
- **Extrapolation**: the surrogate is unreliable for inputs outside the training range of $(\Sigma_a, \Sigma_s, L)$. A Gaussian Process surrogate would provide calibrated uncertainty estimates that grow outside the training domain, making extrapolation risks explicit.
- **No uncertainty quantification**: neither model reports a confidence interval on $\hat{T}$. Bayesian approaches (e.g. Monte Carlo Dropout, Deep Ensembles, or full Gaussian Processes) address this and are better suited to scientific decision-making under uncertainty.

Despite these limitations, the surrogate achieves a speedup of several orders of magnitude relative to direct Monte Carlo, making it practical for tasks such as parameter sweeps, optimisation, or uncertainty propagation that would be prohibitively expensive with repeated simulation.